# Construcción del Frontend para una Aplicación Full-Stack con CrewAI

## Plan Paso a Paso: De la API del Backend a una Página Web

En el notebook anterior (`074-CREWAI-top5-backend.ipynb`), construimos un **backend** — un servidor FastAPI con un endpoint:

```
POST /support/ticket  →  Recibe un ticket, ejecuta CrewAI, devuelve la respuesta de la IA
```

Pero un backend solo es un servidor esperando peticiones. Los usuarios no escriben comandos `curl` — utilizan una **página web**.

**En este notebook, construiremos el frontend** — una página web donde los usuarios pueden:
1. Rellenar un formulario (nombre, email, asunto, mensaje).
2. Hacer clic en "Enviar Ticket".
3. Ver la respuesta de la IA mostrada en la página.

## ¿Qué Es un Frontend?

Un **frontend** es la parte de la aplicación que el usuario ve e interactúa en su navegador. Está construido con HTML, CSS y JavaScript.

Usaremos **Next.js** — el mismo framework de React utilizado en la aplicación de tareas (`051-todoapp-LC-frontend`). Next.js facilita la construcción de páginas web con componentes React.

```
Navegador del Usuario (Frontend)         Servidor (Backend)
┌──────────────────────────┐           ┌──────────────────────┐
│                          │           │                      │
│  ┌────────────────────┐  │  POST     │  FastAPI             │
│  │ Formulario Ticket  │  │ ────────> │    └── CrewAI Flow   │
│  │  Nombre: ________  │  │  JSON     │         ├── Clasificar 
│  │  Email: _______    │  │           │         ├── Enrutar  │
│  │  Asunto: ______    │  │           │         └── Resolver │
│  │  Mensaje: _____    │  │  JSON     │                      │
│  │  [Enviar Ticket]   │  │ <──────── │  Devuelve respuesta  │
│  └────────────────────┘  │           │                      │
│                          │           └──────────────────────┘
│  ┌────────────────────┐  │
│  │ Caja de Respuesta  │  │  ← Muestra la respuesta de la IA
│  │  Categoría: ...    │  │
│  │  Urgencia: ...     │  │
│  │  Mensaje: ...      │  │
│  └────────────────────┘  │
│                          │
└──────────────────────────┘
```

El frontend se ejecuta en `http://localhost:3000` y llama al backend en `http://localhost:8000`.

## La Visión General: Lo Que Estamos Construyendo

Nuestro frontend es simple — solo **una página** con:
- Un **formulario** para enviar un ticket de soporte.
- Un **área de respuesta** que muestra la respuesta de la IA después del envío.

### Estructura del Proyecto

Siguiendo el mismo patrón que el frontend de la aplicación de tareas, nuestra carpeta tendrá este aspecto:

```
074-crewai-frontend/
├── .env                      ← URL del Backend (http://localhost:8000)
├── package.json              ← Dependencias del proyecto (Next.js, React)
├── next.config.mjs           ← Configuración de Next.js
├── jsconfig.json             ← Alias de rutas
├── pages/
│   ├── _app.js               ← Wrapper de la App (carga CSS global)
│   ├── _document.js          ← Estructura del documento HTML
│   └── index.js              ← Página principal (nuestro formulario de tickets)
├── components/
│   ├── layout.js             ← Layout de la página (título + contenedor)
│   ├── ticket-form.js        ← El componente del formulario
│   └── ticket-response.js    ← El componente de visualización de respuesta
└── styles/
    ├── globals.css            ← Estilos globales
    ├── layout.module.css      ← Estilos del layout
    ├── ticket-form.module.css ← Estilos del formulario
    └── ticket-response.module.css ← Estilos de la respuesta
```

**Comparación con la aplicación de tareas:**

| App de Tareas | Nuestra App | Propósito |
|---------------|------------|----------|
| `todo-list.js` | `ticket-form.js` | Componente interactivo principal |
| `todo.js` | `ticket-response.js` | Componente de visualización de resultados |
| `layout.js` | `layout.js` | Wrapper del layout de la página |
| `index.js` | `index.js` | Página principal |

Misma estructura, diferentes componentes.

## Paso 0: Crear el Proyecto Next.js

Primero, necesitamos crear un nuevo proyecto Next.js. Abre un terminal y ejecuta:

```bash
npx create-next-app@latest 074-crewai-frontend
```

Cuando te pregunte, elige estas opciones:
- TypeScript: **No**
- ESLint: **Sí**
- Tailwind CSS: **No**
- Directorio `src/`: **No**
- App Router: **No** (usamos el Pages Router, igual que la app de tareas)
- Alias de importación: **Sí** (mantener el valor predeterminado `@/*`)

Esto crea una nueva carpeta `074-crewai-frontend/` con toda la estructura base de Next.js.

Luego navega a la carpeta:

```bash
cd 074-crewai-frontend
```

## Paso 1: El Archivo `.env`

Al igual que la aplicación de tareas, necesitamos un archivo `.env` para indicar al frontend dónde se está ejecutando el backend.

En Next.js, las variables de entorno que comienzan con `NEXT_PUBLIC_` están disponibles en el código del navegador.

In [ ]:
%%writefile 074-crewai-frontend/.env
NEXT_PUBLIC_API_URL=http://localhost:8000

**Esto es idéntico al archivo `.env` de la aplicación de tareas.** El frontend usará `process.env.NEXT_PUBLIC_API_URL` para construir la URL de las llamadas a la API.

Por ejemplo: `process.env.NEXT_PUBLIC_API_URL + '/support/ticket'` se convierte en `http://localhost:8000/support/ticket`.

## Paso 2: `next.config.mjs` — Configuración de Next.js

Este archivo es igual que el de la aplicación de tareas — solo configuración básica.

In [ ]:
%%writefile 074-crewai-frontend/next.config.mjs
/** @type {import('next').NextConfig} */
const nextConfig = {
  reactStrictMode: true,
};

export default nextConfig;

## Paso 3: `jsconfig.json` — Alias de Rutas

Esto nos permite usar `@/` como atajo para la raíz del proyecto. Por ejemplo, `@/styles/globals.css` en lugar de `../../styles/globals.css`. Igual que la aplicación de tareas.

In [ ]:
%%writefile 074-crewai-frontend/jsconfig.json
{
  "compilerOptions": {
    "paths": {
      "@/*": ["./*"]
    }
  }
}

## Paso 4: `pages/_app.js` — Wrapper de la Aplicación

Este es el componente raíz que envuelve cada página. Su función principal es cargar el CSS global. Idéntico a la aplicación de tareas.

In [ ]:
%%writefile 074-crewai-frontend/pages/_app.js
import "@/styles/globals.css";

export default function App({ Component, pageProps }) {
  return <Component {...pageProps} />;
}

## Paso 5: `pages/_document.js` — Estructura del Documento HTML

Esto define la estructura HTML base. Idéntico a la aplicación de tareas.

In [ ]:
%%writefile 074-crewai-frontend/pages/_document.js
import { Html, Head, Main, NextScript } from "next/document";

export default function Document() {
  return (
    <Html lang="en">
      <Head />
      <body>
        <Main />
        <NextScript />
      </body>
    </Html>
  );
}

## Paso 6: `styles/globals.css` — Estilos Globales

Estilos globales básicos para toda la aplicación. Lo mantenemos simple — solo una fuente limpia y algunos estilos de reinicio.

In [ ]:
%%writefile 074-crewai-frontend/styles/globals.css
* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  background-color: #f5f5f5;
  color: #333;
}

## Paso 7: `styles/layout.module.css` — Estilos del Layout

Estilos para el layout de la página (contenedor centrado con un título). Similar a los estilos del layout de la aplicación de tareas, pero un poco más ancho ya que tenemos un formulario.

In [ ]:
%%writefile 074-crewai-frontend/styles/layout.module.css
.layout {
    max-width: 700px;
    margin: 40px auto;
    padding: 0 20px;
}

.title {
    text-align: center;
    font-size: 28px;
    margin-bottom: 30px;
    color: #1a1a1a;
}

**Comparación con el `layout.module.css` de la aplicación de tareas:**

| App de Tareas | Nuestra App |
|---------------|------------|
| `width: 500px` | `max-width: 700px` (más ancho para el formulario) |
| `margin: 20px` | `margin: 40px auto` (centrado en la página) |

Misma idea, tamaño ligeramente ajustado.

## Paso 8: `styles/ticket-form.module.css` — Estilos del Formulario

Estilos para el formulario de ticket de soporte. Esto es similar a cómo la aplicación de tareas estilizaba su campo de entrada y botones, pero adaptado para un formulario completo.

In [ ]:
%%writefile 074-crewai-frontend/styles/ticket-form.module.css
.form {
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 30px;
    margin-bottom: 20px;
}

.fieldGroup {
    margin-bottom: 15px;
}

.label {
    display: block;
    font-weight: 600;
    margin-bottom: 5px;
    font-size: 14px;
}

.input {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 4px;
    font-size: 14px;
}

.textarea {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 4px;
    font-size: 14px;
    min-height: 100px;
    resize: vertical;
}

.submitBtn {
    width: 100%;
    padding: 12px;
    background-color: #4a90d9;
    color: white;
    border: none;
    border-radius: 4px;
    font-size: 16px;
    cursor: pointer;
    margin-top: 10px;
}

.submitBtn:hover {
    background-color: #357abd;
}

.submitBtn:disabled {
    background-color: #aaa;
    cursor: not-allowed;
}

.loading {
    text-align: center;
    padding: 20px;
    color: #666;
    font-style: italic;
}

## Paso 9: `styles/ticket-response.module.css` — Estilos de Visualización de Respuesta

Estilos para mostrar la respuesta de la IA. Esto es similar al popup de poema de la aplicación de tareas, pero se muestra directamente en la página en lugar de como un modal.

In [ ]:
%%writefile 074-crewai-frontend/styles/ticket-response.module.css
.responseBox {
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 30px;
    margin-top: 20px;
}

.responseTitle {
    font-size: 20px;
    margin-bottom: 15px;
    color: #1a1a1a;
}

.metaRow {
    display: flex;
    gap: 15px;
    margin-bottom: 15px;
    flex-wrap: wrap;
}

.badge {
    padding: 4px 12px;
    border-radius: 12px;
    font-size: 13px;
    font-weight: 600;
}

.badgeSimple {
    background-color: #d4edda;
    color: #155724;
}

.badgeComplex {
    background-color: #fff3cd;
    color: #856404;
}

.badgeLow {
    background-color: #d4edda;
    color: #155724;
}

.badgeMedium {
    background-color: #fff3cd;
    color: #856404;
}

.badgeHigh {
    background-color: #f8d7da;
    color: #721c24;
}

.badgeCategory {
    background-color: #d1ecf1;
    color: #0c5460;
}

.subject {
    font-size: 16px;
    font-weight: 600;
    margin-bottom: 10px;
}

.message {
    line-height: 1.6;
    white-space: pre-wrap;
    background-color: #f9f9f9;
    padding: 15px;
    border-radius: 4px;
    border-left: 3px solid #4a90d9;
}

.followUp {
    margin-top: 15px;
    padding: 10px;
    border-radius: 4px;
    font-size: 14px;
    font-weight: 600;
}

.followUpYes {
    background-color: #fff3cd;
    color: #856404;
}

.followUpNo {
    background-color: #d4edda;
    color: #155724;
}

## Paso 10: `components/layout.js` — Componente de Layout de Página

Este es el wrapper del layout que proporciona un título y centra el contenido en la página. Es el mismo patrón que el `layout.js` de la aplicación de tareas.

In [ ]:
%%writefile 074-crewai-frontend/components/layout.js
import styles from '../styles/layout.module.css'

export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}

**Comparación con el `layout.js` de la aplicación de tareas:**

| App de Tareas | Nuestra App |
|---------------|------------|
| `<h1>To Do</h1>` | `<h1>BigCompany Inc. Support</h1>` |

Mismo componente, diferente título. Eso es todo.

## Paso 11: `components/ticket-response.js` — Componente de Visualización de Respuesta

Este componente recibe los datos de respuesta de la IA y los muestra de forma elegante. Muestra:
- Etiquetas de colores para categoría, complejidad y urgencia.
- El asunto y mensaje de la respuesta.
- Si se necesita seguimiento humano.

En la aplicación de tareas, el componente `todo.js` mostraba un único elemento de tarea. Aquí, `ticket-response.js` muestra la respuesta de la IA.

In [ ]:
%%writefile 074-crewai-frontend/components/ticket-response.js
import styles from '../styles/ticket-response.module.css'

export default function TicketResponse({ response }) {
  // Elegir la clase CSS correcta según el valor
  const complexityClass = response.complexity === 'simple'
    ? styles.badgeSimple
    : styles.badgeComplex;

  const urgencyClass = response.urgency === 'low'
    ? styles.badgeLow
    : response.urgency === 'medium'
      ? styles.badgeMedium
      : styles.badgeHigh;

  return (
    <div className={styles.responseBox}>
      <h2 className={styles.responseTitle}>AI Response</h2>

      {/* Etiquetas de colores para categoría, complejidad, urgencia */}
      <div className={styles.metaRow}>
        <span className={`${styles.badge} ${styles.badgeCategory}`}>
          {response.category}
        </span>
        <span className={`${styles.badge} ${complexityClass}`}>
          {response.complexity} path
        </span>
        <span className={`${styles.badge} ${urgencyClass}`}>
          {response.urgency} urgency
        </span>
      </div>

      {/* Asunto y mensaje de la respuesta */}
      <div className={styles.subject}>{response.resolution_subject}</div>
      <div className={styles.message}>{response.resolution_message}</div>

      {/* Indicador de seguimiento */}
      <div className={`${styles.followUp} ${
        response.follow_up_needed ? styles.followUpYes : styles.followUpNo
      }`}>
        {response.follow_up_needed
          ? 'Human follow-up needed'
          : 'Fully resolved by AI — no follow-up needed'}
      </div>
    </div>
  )
}

**Cómo se compara esto con el `todo.js` de la aplicación de tareas:**

| App de Tareas (`todo.js`) | Nuestra App (`ticket-response.js`) |
|--------------------------|------------------------------------|
| Recibe una prop `todo` | Recibe una prop `response` |
| Muestra checkbox + campo de texto | Muestra etiquetas + mensaje |
| Tiene botón "Generate Poem" (llama al backend) | Sin acciones extra — solo muestra datos |
| Usa portal para popup de poema | Sin popup — visualización en línea |

Nuestro componente es más simple: solo recibe datos y los muestra. No se necesita interacción del usuario.

## Paso 12: `components/ticket-form.js` — El Componente del Formulario

Este es el componente interactivo principal — el corazón de nuestro frontend. Gestiona:
1. **Estado del formulario** — seguimiento de lo que el usuario escribe en cada campo.
2. **Envío del ticket** — envío de los datos al backend.
3. **Estado de carga** — muestra "Procesando..." mientras la IA trabaja.
4. **Visualización de la respuesta** — pasa el resultado al componente `TicketResponse`.

En la aplicación de tareas, `todo-list.js` era el componente principal. Gestionaba:
- Añadir tareas (POST al backend)
- Obtener tareas (GET del backend)
- Actualizar tareas (PUT al backend)
- Eliminar tareas (DELETE al backend)

Nuestro componente es más simple — solo hace una cosa: **enviar un ticket mediante POST y mostrar la respuesta**.

In [ ]:
%%writefile 074-crewai-frontend/components/ticket-form.js
import styles from '../styles/ticket-form.module.css'
import { useState } from 'react'
import TicketResponse from './ticket-response'

export default function TicketForm() {
  // Campos del formulario — misma idea que el estado mainInput de la app de tareas
  const [name, setName] = useState('')
  const [email, setEmail] = useState('')
  const [subject, setSubject] = useState('')
  const [message, setMessage] = useState('')

  // Estado de carga y respuesta
  const [loading, setLoading] = useState(false)
  const [response, setResponse] = useState(null)

  // Enviar el ticket al backend
  async function handleSubmit(e) {
    e.preventDefault()  // Evitar que la página se recargue

    setLoading(true)
    setResponse(null)

    // POST al backend — mismo patrón que la función addToDo de la app de tareas
    const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/support/ticket', {
      method: 'POST',
      body: JSON.stringify({
        customer_name: name,
        customer_email: email,
        ticket_subject: subject,
        ticket_message: message,
      }),
      headers: {
        'Content-Type': 'application/json'
      }
    })

    if (res.ok) {
      const data = await res.json()
      setResponse(data)
    }

    setLoading(false)
  }

  return (
    <div>
      {/* El formulario del ticket */}
      <form className={styles.form} onSubmit={handleSubmit}>
        <div className={styles.fieldGroup}>
          <label className={styles.label}>Your Name</label>
          <input
            className={styles.input}
            type="text"
            value={name}
            onChange={(e) => setName(e.target.value)}
            placeholder="e.g. Alice Johnson"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Your Email</label>
          <input
            className={styles.input}
            type="email"
            value={email}
            onChange={(e) => setEmail(e.target.value)}
            placeholder="e.g. alice@example.com"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Subject</label>
          <input
            className={styles.input}
            type="text"
            value={subject}
            onChange={(e) => setSubject(e.target.value)}
            placeholder="e.g. Can't reset my password"
            required
          />
        </div>

        <div className={styles.fieldGroup}>
          <label className={styles.label}>Message</label>
          <textarea
            className={styles.textarea}
            value={message}
            onChange={(e) => setMessage(e.target.value)}
            placeholder="Describe your issue in detail..."
            required
          />
        </div>

        <button
          className={styles.submitBtn}
          type="submit"
          disabled={loading}
        >
          {loading ? 'Processing...' : 'Submit Ticket'}
        </button>
      </form>

      {/* Mensaje de carga */}
      {loading && (
        <div className={styles.loading}>
          The AI crew is working on your ticket. This may take a minute...
        </div>
      )}

      {/* Mostrar la respuesta de la IA */}
      {response && <TicketResponse response={response} />}
    </div>
  )
}

### Cómo se Compara la Llamada `fetch` con la de la App de Tareas

El patrón central es idéntico. Comparemos la llamada `fetch` en ambas aplicaciones:

**Aplicación de tareas** (`todo-list.js` — función `addToDo`):
```javascript
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/todos/', {
  method: 'POST',
  body: JSON.stringify({
    name: name,
    completed: false
  }),
  headers: {
    'Content-Type': 'application/json'
  }
})
```

**Nuestra aplicación** (`ticket-form.js` — función `handleSubmit`):
```javascript
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/support/ticket', {
  method: 'POST',
  body: JSON.stringify({
    customer_name: name,
    customer_email: email,
    ticket_subject: subject,
    ticket_message: message,
  }),
  headers: {
    'Content-Type': 'application/json'
  }
})
```

Misma estructura:
1. `fetch()` con la URL del backend + ruta del endpoint.
2. `method: 'POST'` para enviar datos.
3. `body: JSON.stringify(...)` para convertir nuestros datos a JSON.
4. `headers` para indicar al backend que estamos enviando JSON.

La única diferencia es la URL (`/todos/` vs `/support/ticket`) y los campos de datos.

## Paso 13: `pages/index.js` — La Página Principal

Esta es la página principal de nuestra aplicación. Une todo:
- Usa el componente `Layout` para la estructura de la página.
- Usa el componente `TicketForm` para el formulario y la respuesta.

Esto sigue exactamente el mismo patrón que el `index.js` de la aplicación de tareas.

In [ ]:
%%writefile 074-crewai-frontend/pages/index.js
import Head from 'next/head'
import Layout from '../components/layout'
import TicketForm from '../components/ticket-form'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Support</title>
        <meta name="description" content="AI-Powered Customer Support" />
        <link rel="icon" href="/favicon.ico" />
      </Head>
      <Layout>
        <TicketForm />
      </Layout>
    </div>
  )
}

**Comparación con el `index.js` de la aplicación de tareas:**

```javascript
// Aplicación de tareas
import ToDoList from '../components/todo-list'
<Layout>
  <ToDoList />
</Layout>

// Nuestra aplicación
import TicketForm from '../components/ticket-form'
<Layout>
  <TicketForm />
</Layout>
```

Estructura idéntica. Simplemente intercambiamos `ToDoList` por `TicketForm`.

## Paso 14: Cómo Ejecutar el Frontend

**Requisitos previos:** Asegúrate de tener Node.js instalado y de que el backend esté en ejecución (`uvicorn main:app --reload` en la carpeta `074-crewai-backend/`).

Luego, en un **nuevo terminal**, navega a la carpeta `074-crewai-frontend/` y ejecuta:

```bash
npm install
npm run dev
```

Esto inicia el frontend en `http://localhost:3000`.

Deberías ver:

```
  ▲ Next.js
  - Local:   http://localhost:3000
```

Abre `http://localhost:3000` en tu navegador y verás el formulario de ticket de soporte.

## Paso 15: Ejecutar la Aplicación Full Stack

Para ejecutar la aplicación completa, necesitas **dos terminales** ejecutándose al mismo tiempo:

### Terminal 1 — Backend (puerto 8000)
```bash
cd 074-crewai-backend
uvicorn main:app --reload
```

### Terminal 2 — Frontend (puerto 3000)
```bash
cd 074-crewai-frontend
npm run dev
```

Luego abre `http://localhost:3000` en tu navegador.

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│  Terminal 1 (Backend)      Terminal 2 (Frontend)    │
│  ┌──────────────────┐      ┌──────────────────┐     │
│  │ uvicorn main:app │      │ npm run dev      │     │
│  │ --reload         │      │                  │     │
│  │                  │      │                  │     │
│  │ Ejecutándose en  │      │ Ejecutándose en  │     │
│  │ localhost:8000   │ ◄──► │ localhost:3000   │     │
│  └──────────────────┘      └──────────────────┘     │
│                                                     │
│  Navegador: http://localhost:3000                   │
│                                                     │
└─────────────────────────────────────────────────────┘
```

### Cómo probarlo:

1. Rellena el formulario:
   - Nombre: `Alice Johnson`
   - Email: `alice@example.com`
   - Asunto: `Can't log in — need password reset`
   - Mensaje: `Hi, I forgot my password and can't log into my account. How do I reset it?`

2. Haz clic en **Submit Ticket**.

3. Espera a que la IA procese (esto tarda un minuto — el CrewAI Flow se está ejecutando en el backend).

4. Observa cómo la respuesta de la IA aparece debajo del formulario con:
   - Etiqueta de categoría (p. ej., `password_reset`)
   - Etiqueta de complejidad (p. ej., `simple path`)
   - Etiqueta de urgencia (p. ej., `low urgency`)
   - El mensaje completo de la respuesta
   - Si se necesita seguimiento humano o no

## Resumen: La Arquitectura Full-Stack Completa

Así es como todo encaja:

```
┌──────────────────────────────────────────────────────────────────────┐
│  FRONTEND (Next.js — localhost:3000)                                 │
│                                                                      │
│  pages/index.js                                                      │
│    └── components/layout.js          ← Wrapper de la página (título) │
│    └── components/ticket-form.js     ← Formulario + fetch() al backend
│          └── components/ticket-response.js  ← Muestra respuesta IA   │
│                                                                      │
│  .env → NEXT_PUBLIC_API_URL=http://localhost:8000                    │
└──────────────────────────────┬───────────────────────────────────────┘
                               │
                    POST /support/ticket
                         (JSON)
                               │
                               ▼
┌──────────────────────────────────────────────────────────────────────┐
│  BACKEND (FastAPI — localhost:8000)                                  │
│                                                                      │
│  main.py                             ← App FastAPI + CORS            │
│    └── routers/support.py            ← Endpoint POST /support/ticket │
│          └── crew_logic.py           ← CrewAI Flow + Agentes         │
│                ├── Agente Clasificador ← Clasifica el ticket         │
│                ├── Agente FAQ          ← Responde preguntas simples  │
│                ├── Agente Investigador ← Investiga problemas complejos
│                └── Agente Solución     ← Escribe respuestas empáticas│
│                                                                      │
│  .env → OPENAI_API_KEY=sk-...                                        │
└──────────────────────────────────────────────────────────────────────┘
```

### Resumen Archivo por Archivo

| Archivo | Qué Hace | Inspirado En |
|---------|----------|-------------|
| `.env` | URL del Backend | `051-todoapp-LC-frontend/.env` |
| `next.config.mjs` | Configuración de Next.js | `051-todoapp-LC-frontend/next.config.mjs` |
| `jsconfig.json` | Alias de rutas | `051-todoapp-LC-frontend/jsconfig.json` |
| `pages/_app.js` | Wrapper de la app, carga CSS | `051-todoapp-LC-frontend/pages/_app.js` |
| `pages/_document.js` | Estructura HTML | `051-todoapp-LC-frontend/pages/_document.js` |
| `pages/index.js` | Página principal | `051-todoapp-LC-frontend/pages/index.js` |
| `components/layout.js` | Wrapper del layout | `051-todoapp-LC-frontend/components/layout.js` |
| `components/ticket-form.js` | Formulario + llamada API | `051-todoapp-LC-frontend/components/todo-list.js` |
| `components/ticket-response.js` | Visualización de respuesta | `051-todoapp-LC-frontend/components/todo.js` |
| `styles/*.module.css` | Estilos de componentes | `051-todoapp-LC-frontend/styles/*.module.css` |

## Conclusiones Clave para Principiantes

1. **El frontend y el backend son programas separados.** El frontend se ejecuta en el puerto 3000 (Next.js), el backend se ejecuta en el puerto 8000 (FastAPI). Se comunican entre sí mediante peticiones HTTP (JSON).

2. **`fetch()` es cómo el frontend habla con el backend.** Envía una petición POST con los datos del ticket y recibe la respuesta de la IA. Es el mismo patrón de `fetch()` utilizado en la aplicación de tareas.

3. **CORS hace esto posible.** Sin la configuración de CORS en el backend, el navegador bloquearía las peticiones de `localhost:3000` a `localhost:8000`. Por eso `main.py` incluye `CORSMiddleware`.

4. **Los archivos `.env` almacenan la configuración.** El frontend usa `NEXT_PUBLIC_API_URL` para saber dónde está el backend. El backend usa `OPENAI_API_KEY` para autenticarse con OpenAI.

5. **El estado de React (`useState`) rastrea lo que el usuario escribe.** Cada campo del formulario tiene su propia variable de estado. Cuando el usuario escribe, el estado se actualiza. Cuando envía, los valores del estado se envían al backend.

6. **Los componentes son bloques de construcción reutilizables.** `Layout` envuelve la página, `TicketForm` gestiona el formulario, `TicketResponse` muestra el resultado. Cada componente tiene su propio módulo CSS.

7. **La estructura del proyecto sigue el mismo patrón que la aplicación de tareas.** Si entendiste la aplicación de tareas, ya entiendes el 90% de esta aplicación. Lo único nuevo es el formulario y la visualización de la respuesta.